# Demand Forecasting (LightGBM) — clean, error-free version

Interac e-Transfer **transaction-volume** forecasting. Run the cells top to bottom.

This version removes the things that were causing errors:
- reads data with **`wslib`** (project token) instead of hard-coded COS keys — no bucket / region-endpoint problems;
- no deprecated `project_lib` calls (`project.get_metadata()` / `project.save_data`);
- one clean `ibm_watsonx_ai` client for deploy + score (no mixed SDKs, no manual IAM token).

> Fill in only three values: **URL** (region), **API_KEY**, **SPACE_ID**. Everything else runs as-is.

## 0. Insert your project token first
Top-right **⋮ (More) → Insert project token**, then run that inserted cell. It defines `project` and `wslib`. The check below confirms it worked.

In [ ]:
try:
    wslib
except NameError:
    raise RuntimeError(
        "Run the 'Insert project token' cell first: top-right \u22ee (More) \u2192 Insert project token, then run it."
    )
print("Project token OK \u2014 wslib is ready.")

## 1. Install packages (run once)
If it says *restart the kernel*, restart, then re-run the project-token cell and continue from the top.

In [ ]:
%pip install -q "numpy<2" pandas matplotlib lightgbm scikit-learn scipy pyarrow

In [ ]:
import io
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping

## 2. Credentials + deployment space
- **URL**: match your region. Toronto \u2192 `https://ca-tor.ml.cloud.ibm.com`; Dallas \u2192 `https://us-south.ml.cloud.ibm.com`.
- **API_KEY**: IBM Cloud API key (Manage \u2192 Access (IAM) \u2192 API keys).
- **SPACE_ID**: your deployment space GUID (Deployment spaces \u2192 your space \u2192 Manage \u2192 General). Must be in the **same account + region** as URL.

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials

URL      = "https://ca-tor.ml.cloud.ibm.com"   # Toronto (Dallas: https://us-south.ml.cloud.ibm.com)
API_KEY  = "PASTE_YOUR_API_KEY"
SPACE_ID = "PASTE_YOUR_SPACE_ID"

client = APIClient(Credentials(url=URL, api_key=API_KEY))
client.set.default_space(SPACE_ID)
print("Connected. Default space:", SPACE_ID)

## 3. Load training data
Reads the `training_data_v2.csv` data asset from the project via `wslib` \u2014 no COS keys, no bucket names, no region endpoint to get wrong. (Make sure the file is added to this project first.)

In [ ]:
raw = wslib.load_data("training_data_v2.csv")
df_1 = pd.read_csv(raw if hasattr(raw, "read") else io.BytesIO(raw))
print("Loaded:", df_1.shape)
df_1.head()

## 4. Clean + prepare

In [ ]:
if len(df_1) > 50000:
    df_1 = df_1.iloc[50000:]

# This dataset has no date column — every column is a numeric feature or the target.
df = df_1.apply(pd.to_numeric, errors="coerce").fillna(0.0)
if "SEGMENT_ID" in df.columns:
    df = df.sort_values("SEGMENT_ID")
df = df.reset_index(drop=True)
print(df.shape)
df.head()

## 5. Train / validation / test split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

split_year = X["year"].quantile(0.8)
mask = X["year"] <= split_year
X_train, X_val = X[mask], X[~mask]
y_train, y_val = y[mask], y[~mask]

X_test, y_test = X.tail(1000), y.tail(1000)
print("train / val / test:", X_train.shape, X_val.shape, X_test.shape)

## 6. (Optional) save train/test back to the project
Uses `wslib.save_data` (the modern replacement for `project.save_data`). Safe to skip.

In [ ]:
train_df = pd.concat([X_train, y_train.rename("target")], axis=1)
test_df  = pd.concat([X_test,  y_test.rename("target")], axis=1)
try:
    wslib.save_data("training_data.csv", train_df.to_csv(index=False).encode(), overwrite=True)
    wslib.save_data("test_data.csv",     test_df.to_csv(index=False).encode(),  overwrite=True)
    print("Saved training_data.csv and test_data.csv to the project.")
except Exception as e:
    print("Skipped saving to project:", e)

## 7. Train the LightGBM model

In [ ]:
def train_lgbm_model(x_train, y_train, x_valid, y_valid):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42,
        metric="rmse",
    )
    model.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[early_stopping(50)],
    )
    return model

m_lgb = train_lgbm_model(X_train, y_train, X_val, y_val)

## 8. Evaluation report
**Note the RMSE** \u2014 you enter it as the OpenScale **Quality** RMSE threshold in the lab's Part B.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

y_val_pred = m_lgb.predict(X_val)
yt = np.asarray(y_val, dtype=float)
yp = np.asarray(y_val_pred, dtype=float)

rmse = mean_squared_error(yt, yp) ** 0.5
mae  = mean_absolute_error(yt, yp)
r2   = r2_score(yt, yp)
pear = pearsonr(yt, yp)[0]
spear = spearmanr(yt, yp)[0]

nz = yt != 0                       # avoid divide-by-zero in MAPE
mape = float(np.mean(np.abs((yt[nz] - yp[nz]) / yt[nz])) * 100) if nz.any() else float("nan")

print(f"RMSE     : {rmse:,.2f}   <-- use this as the OpenScale Quality RMSE threshold")
print(f"MAE      : {mae:,.2f}")
print(f"R2       : {r2:.3f}")
print(f"Pearson  : {pear:.3f}")
print(f"Spearman : {spear:.3f}")
print(f"MAPE(%)  : {mape:.2f}")

## 9. Register + deploy the model

In [ ]:
software_spec_id = client.software_specifications.get_id_by_name("runtime-24.1-py3.11")

model_meta = {
    client.repository.ModelMetaNames.NAME: "demand_forecasting_lgbm",
    client.repository.ModelMetaNames.TYPE: "scikit-learn_1.3",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
}
model_details = client.repository.store_model(model=m_lgb, meta_props=model_meta)
model_id = client.repository.get_model_id(model_details)
print("Model ID:", model_id)

dep_meta = {
    client.deployments.ConfigurationMetaNames.NAME: "demand-forecasting-ml",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}
dep_details = client.deployments.create(model_id, meta_props=dep_meta)
deployment_id = client.deployments.get_id(dep_details)
print("Deployment SUCCESSFUL \u2014 Deployment ID:", deployment_id)

## 10. (Optional) score the deployed model
Uses the SDK and a real row from the test set, so the columns always match \u2014 no hard-coded fields or IAM token.

In [ ]:
sample = X_test.iloc[[0]]
payload = {"input_data": [{"fields": sample.columns.tolist(), "values": sample.values.tolist()}]}
print(client.deployments.score(deployment_id, payload))